In [41]:
from IPython.display import Markdown, display
import jax.numpy as jnp
from jax import value_and_grad
from jax import hessian
from jax import random
from scipy.optimize import minimize

In [2]:
def to_latex_matrix(matrix):
    """Convert a JAX/numpy matrix to a LaTeX bmatrix string."""
    rows = []
    for row in matrix:
        rows.append(" & ".join(f"{val:.2f}" for val in row))
    return r"\begin{bmatrix}" + r" \\ ".join(rows) + r"\end{bmatrix}"

def to_latex_vector(vector):
    """Convert a JAX/numpy vector to a LaTeX bmatrix column vector."""
    rows = r" \\ ".join(f"{val:.2f}" for val in vector)
    return r"\begin{bmatrix}" + rows + r"\end{bmatrix}"

# Gradient and hessian

The gradient and the Hessian are given by

$$\begin{align*}
\nabla_{\mathbf{f}} \log p(\mathbf{y}, \mathbf{f}) &= \sum_{n=1}^N \nabla_{\mathbf{f}}\log  p(y_n|f_n) - \frac12 \nabla_{\mathbf{f}}\mathbf{f}^T\mathbf{K}^{-1} \mathbf{f} = \mathbf{g} - \mathbf{K}^{-1} \mathbf{f},\\
%
\nabla_{\mathbf{f}}^2 \log p(\mathbf{y}, \mathbf{f}) &= \sum_{n=1}^N  \nabla_{\mathbf{f}}^2 \log  p(y_n|f_n) - \frac12 \nabla_{\mathbf{f}}^2\mathbf{f}^T\mathbf{K}^{-1} \mathbf{f}
=\sum_{n=1}^N  \nabla_{\mathbf{f}}^2 \log  p(y_n|f_n) - \mathbf{K}^{-1} 
=-\Lambda - \mathbf{K}^{-1},
\end{align*}$$

where $\mathbf{g} \in \mathbb{R}^N$ with entries given by $g_n = \frac{\partial}{\partial f_n}\log  p(y_n|f_n)$ and $\Lambda \in \mathbb{R}^{N \times N}$ is a diagonal matrix with elements $\Lambda_{nn} = -\nabla_{\mathbf{f}}^2 \log  p(y_n|f_n)$ evaluated at the mode $\hat{\mathbf{f}}_{\text{MAP}} = \argmax_{\mathbf{f}} \log p(\mathbf{y}, \mathbf{f})$.

## Gradient code

In [ ]:
def compute_grad(log_joint):
    grad = value_and_grad(log_joint)
    
    return grad

def validate_grad(grad_fun, num_params, W_map):
    result = minimize(grad_fun, w_init_flat = jnp.zeros(num_params), jac=True)

    if result.success:
        w_MAP_found = result.x
        if w_MAP_found == W_map:
            print("Grad function is correct")
        else:
            print("W_map found with grad function does not equal the given W_map")
    else:
        print("ERROR")

compute_grad(log_joint)

<function __main__.log_joint(w_flat)>

## Hessian code

In [40]:
def compute_hessian(log_joint, w_map):
    
    H = hessian(log_joint)(w_map)

    return H

def evaluate_hessian(H, W_map):
    return H(W_map.flatten())